In [ ]:
import pandas as pd
import warnings
import csv
warnings.filterwarnings("ignore", category=DeprecationWarning)

The parse tree server requires some changes to the keystroke csv formatting. Using this code to prepare the keystrokes, and load this version into KeystrokeExplorer

It may take a while to run.

In [ ]:
# Change fileName to be the keystroke csv file,
# outFileName should be what you want the file to be called
fileName = ""
outFileName = ""

df = pd.read_csv(fileName)
df.SourceLocation = df.SourceLocation.fillna(0)


In [ ]:
def reconstruct(df):
    s = ''
    for _,row in df[df.EventType.isin(["File.Edit","X-FileInit"])].iterrows():
        i = int(row.SourceLocation)
        insert = '' if pd.isna(row.InsertText) else row.InsertText
        delete = '' if pd.isna(row.DeleteText) else row.DeleteText
        s = s[:i] + insert + s[i+len(delete):]
    return s

def is_compilable(code: str) -> bool:
    try:
        compile(code, "", "exec")
        return 1
    except SyntaxError:
        return 0
    
def add_compilable_column(df):
    current_code = {}
    compilable_flags = []

    for i, row in df.iterrows():
        compilable = None
        if row.EventType in ["File.Edit","X-FileInit"]:
            file_id = row.FileID  

            if file_id not in current_code:
                current_code[file_id] = ''

            s = current_code[file_id]

            i = int(row.SourceLocation)
            insert = '' if pd.isna(row.InsertText) else row.InsertText
            delete = '' if pd.isna(row.DeleteText) else row.DeleteText

            s = s[:i] + insert + s[i + len(delete):]
            current_code[file_id] = s

            compilable = is_compilable(s)

        compilable_flags.append(compilable)

    df["X-Compilable"] = compilable_flags
    return df

df["FileID"] = df.apply(lambda x: f"{x.SubjectID}_{x.AssignmentID}_{x.CodeStateSection}", axis=1)
df = add_compilable_column(df)


In [ ]:
df.to_csv(outFileName, quoting=csv.QUOTE_NONNUMERIC)